# Module 04 — Lecture 1: The Hodgkin-Huxley Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_04_hodgkin_huxley/01_hh_theory.ipynb)

---

Alan Hodgkin and Andrew Huxley recorded action potentials from the giant squid axon in 1952 and described ion channel dynamics with a set of coupled ODEs that won them the Nobel Prize in 1963. This model remains the foundation of modern computational neuroscience.

**Learning objectives:**
- Understand the biological basis of the HH equations
- Implement each voltage-dependent rate function
- Simulate a single HH neuron in Python (CPU)
- Visualise the action potential and ion channel kinetics
- Compare to the LIF model

## 1. Biological Background

The action potential is generated by the coordinated opening and closing of **voltage-gated ion channels**:

```
Resting state (V ≈ -65 mV):
  • Na⁺ channels CLOSED — maintained by inactivation gate (h)
  • K⁺ channels mostly CLOSED
  • Net: small leak current balances

Depolarisation (V crosses threshold ~-55 mV):
  • Na⁺ activation gate m opens FAST (τ ≈ 0.3 ms)
  • Na⁺ rushes in → V spikes to +40 mV

Repolarisation:
  • Na⁺ inactivation gate h closes (τ ≈ 1 ms) → Na⁺ current stops
  • K⁺ activation gate n opens SLOW (τ ≈ 4 ms) → K⁺ rushes out
  • V returns toward EK ≈ -77 mV (hyperpolarisation)

Refractory period:
  • h still closed → Na⁺ unavailable → cannot fire again
  • K⁺ channels slowly close → system returns to rest
```

## 2. The HH Equations

$$C_m \frac{dV}{dt} = -g_{Na} m^3 h (V - E_{Na}) - g_K n^4 (V - E_K) - g_L (V - E_L) + I_{ext}$$

$$\frac{dm}{dt} = \alpha_m(V)(1-m) - \beta_m(V)m$$
$$\frac{dh}{dt} = \alpha_h(V)(1-h) - \beta_h(V)h$$  
$$\frac{dn}{dt} = \alpha_n(V)(1-n) - \beta_n(V)n$$

**State variables:** V (membrane voltage), m (Na activation), h (Na inactivation), n (K activation)

**Parameters (Hodgkin-Huxley 1952 values):**

| Parameter | Value | Unit | Meaning |
|-----------|-------|------|--------|
| $C_m$ | 1.0 | μF/cm² | membrane capacitance |
| $g_{Na}$ | 120 | mS/cm² | max Na conductance |
| $g_K$ | 36 | mS/cm² | max K conductance |
| $g_L$ | 0.3 | mS/cm² | leak conductance |
| $E_{Na}$ | +50 | mV | Na reversal potential |
| $E_K$ | -77 | mV | K reversal potential |
| $E_L$ | -54.4 | mV | leak reversal |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# HH parameters
Cm  = 1.0;  gNa = 120.0; gK = 36.0; gL = 0.3
ENa = 50.0; EK  = -77.0; EL = -54.4

# Rate functions
def alpha_m(V): 
    dv = V + 40
    return 1.0 if abs(dv) < 1e-5 else 0.1 * dv / (1 - np.exp(-dv/10))

def beta_m(V):  return 4.0 * np.exp(-(V + 65) / 18)
def alpha_h(V): return 0.07 * np.exp(-(V + 65) / 20)
def beta_h(V):  return 1.0 / (1 + np.exp(-(V + 35) / 10))

def alpha_n(V): 
    dv = V + 55
    return 0.1 if abs(dv) < 1e-5 else 0.01 * dv / (1 - np.exp(-dv/10))

def beta_n(V):  return 0.125 * np.exp(-(V + 65) / 80)

# Steady-state gating variables at rest
V_rest = -65.0
m_inf = lambda V: alpha_m(V) / (alpha_m(V) + beta_m(V))
h_inf = lambda V: alpha_h(V) / (alpha_h(V) + beta_h(V))
n_inf = lambda V: alpha_n(V) / (alpha_n(V) + beta_n(V))

# Visualise steady-state activation curves
V_range = np.linspace(-100, 60, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(V_range, [m_inf(v)**3 for v in V_range], 'b', lw=2, label='m³ (Na activation)')
ax1.plot(V_range, [h_inf(v) for v in V_range], 'r', lw=2, label='h (Na inactivation)')
ax1.plot(V_range, [n_inf(v)**4 for v in V_range], 'g', lw=2, label='n⁴ (K activation)')
ax1.set_xlabel('Voltage (mV)', fontsize=12)
ax1.set_ylabel('Steady-state value', fontsize=12)
ax1.set_title('HH Steady-State Activation Curves', fontsize=13)
ax1.legend(fontsize=11); ax1.grid(True, alpha=0.3)
ax1.axvline(V_rest, color='gray', linestyle=':', alpha=0.7, label='V_rest')

# Time constants
tau_m = lambda V: 1/(alpha_m(V) + beta_m(V))
tau_h = lambda V: 1/(alpha_h(V) + beta_h(V))
tau_n = lambda V: 1/(alpha_n(V) + beta_n(V))

ax2.plot(V_range, [tau_m(v) for v in V_range], 'b', lw=2, label='τ_m (Na act.)')
ax2.plot(V_range, [tau_h(v) for v in V_range], 'r', lw=2, label='τ_h (Na inact.)')
ax2.plot(V_range, [tau_n(v) for v in V_range], 'g', lw=2, label='τ_n (K act.)')
ax2.set_xlabel('Voltage (mV)', fontsize=12)
ax2.set_ylabel('Time constant (ms)', fontsize=12)
ax2.set_title('HH Channel Time Constants', fontsize=13)
ax2.legend(fontsize=11); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hh_kinetics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Resting state: m={m_inf(V_rest):.4f}, h={h_inf(V_rest):.4f}, n={n_inf(V_rest):.4f}")

In [ ]:
# Simulate single HH neuron — CPU implementation
def simulate_hh(I_ext_fn, T_ms=100, dt=0.01):
    T = int(T_ms / dt)
    t = np.arange(T) * dt
    V = np.zeros(T); m = np.zeros(T); h = np.zeros(T); n = np.zeros(T)

    # Initial conditions at rest
    V[0] = V_rest
    m[0] = m_inf(V_rest); h[0] = h_inf(V_rest); n[0] = n_inf(V_rest)

    for i in range(1, T):
        v, mi, hi, ni = V[i-1], m[i-1], h[i-1], n[i-1]
        I = I_ext_fn(t[i-1])

        I_Na = gNa * mi**3 * hi * (v - ENa)
        I_K  = gK  * ni**4 * (v - EK)
        I_L  = gL  * (v - EL)

        dV = (I - I_Na - I_K - I_L) / Cm
        dm = alpha_m(v)*(1-mi) - beta_m(v)*mi
        dh = alpha_h(v)*(1-hi) - beta_h(v)*hi
        dn = alpha_n(v)*(1-ni) - beta_n(v)*ni

        V[i] = v  + dt * dV
        m[i] = mi + dt * dm
        h[i] = hi + dt * dh
        n[i] = ni + dt * dn

    return t, V, m, h, n

# Current step: 10 μA/cm² from t=10 ms to t=90 ms
I_fn = lambda t: 10.0 if 10 <= t <= 90 else 0.0
t, V, m, h, n = simulate_hh(I_fn, T_ms=100, dt=0.01)

# Full HH action potential visualisation
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Voltage
axes[0].plot(t, V, 'k', lw=1.5)
axes[0].set_ylabel('V (mV)', fontsize=12)
axes[0].set_title('Hodgkin-Huxley Action Potential (I = 10 μA/cm², 10-90 ms)', fontsize=13)
axes[0].axhline(ENa, color='b', linestyle=':', alpha=0.4, label=f'E_Na = {ENa} mV')
axes[0].axhline(EK,  color='r', linestyle=':', alpha=0.4, label=f'E_K = {EK} mV')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

# Gating variables
axes[1].plot(t, m, 'b', lw=1.5, label='m (Na activation)')
axes[1].plot(t, h, 'r', lw=1.5, label='h (Na inactivation)')
axes[1].plot(t, n, 'g', lw=1.5, label='n (K activation)')
axes[1].set_ylabel('Gating variable', fontsize=12)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

# Ion currents
I_Na = gNa * m**3 * h * (V - ENa)
I_K  = gK  * n**4 * (V - EK)
I_L  = gL  * (V - EL)
axes[2].plot(t, -I_Na, 'b', lw=1.5, label='−I_Na (inward)')
axes[2].plot(t,  I_K,  'r', lw=1.5, label='I_K (outward)')
axes[2].plot(t,  I_L,  'g', lw=1, label='I_L (leak)', alpha=0.7)
axes[2].set_ylabel('Current (μA/cm²)', fontsize=12)
axes[2].set_xlabel('Time (ms)', fontsize=12)
axes[2].legend(fontsize=10); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hh_action_potential.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. HH vs LIF: What's Different?

| Feature | LIF | Hodgkin-Huxley |
|---------|-----|----------------|
| State variables | 1 (V) | 4 (V, m, h, n) |
| Spike mechanism | Threshold rule | Emergent from ion channels |
| Refractory period | Explicit counter | Emergent from h inactivation |
| Action potential shape | Instantaneous (artificial) | Realistic ~1 ms spike |
| Compute per step | ~5 FLOPs | ~50 FLOPs |
| Required dt | 0.1 ms OK | 0.01 ms needed |
| Biological realism | Low | High |

**Key insight for GPU implementation:** Each neuron still needs only one thread. But now each thread computes ~50 FLOPs instead of ~5, so the kernel becomes more compute-bound and benefits from better arithmetic throughput. The critical optimisation is using `__device__` functions efficiently.

**Next lecture:** GPU implementation with both Euler and RK4 solvers.